In [ ]:
%pip install langchain openai langchain-core langchain-openai langchain-community

In [ ]:
import os
from dotenv import load_dotenv

# --- Load API Key ---
load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
property_info = """
    123 Main Street, Austin TX

    25,000 SF Retail Center
    95% Occupied
    Anchored by Starbucks
    Average Household Income: $145,000
    Traffic Count: 42,000 VPD
    Price: $8.5M
"""

In [ ]:
from langchain_openai import ChatOpenAI
# from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate

# Generate property descriptioin using LLM and prompt template
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

In [14]:
llm = ChatOpenAI(model="gpt-4o-mini")

listing_prompt = PromptTemplate(
    input_variables=["property"],
    template="""
        You are a commercial real estate broker.

        Create a professional CRE listing description.

        Property Information:
        {property}
    """
)

chain = listing_prompt | llm

result = chain.invoke({
    "property": property_info
})

print(result.content)

### Prime Retail Investment Opportunity in Austin, TX

**Property Address:**  
123 Main Street, Austin, TX

**Listing Price:**  
$8,500,000

**Property Overview:**  
We are pleased to present an exceptional investment opportunity in the heart of Austin, Texas, featuring a 25,000 square foot retail center strategically located at 123 Main Street. This property boasts a remarkable occupancy rate of 95% and is anchored by the internationally recognized brand, Starbucks, ensuring a steady flow of foot traffic and consistent customer engagement.

**Key Features:**
- **Total Square Footage:** 25,000 SF
- **Occupancy Rate:** 95%
- **Anchor Tenant:** Starbucks
- **Average Household Income:** $145,000
- **Daily Traffic Count:** 42,000 VPD

This retail center serves a high-income demographic with an average household income of $145,000, making it an ideal location for various retail, dining, and service-oriented businesses. The site benefits from a substantial daily traffic count of 42,000 vehic

In [11]:
#Let's create a LinkedIn post for the above listing using the generated description
linkedin_prompt = PromptTemplate(
    input_variables=["listing"],
    template="""
        Create a professional LinkedIn post for commercial real estate investors.

        Listing Description:
        {listing}

        Include:
        - Hook
        - Key highlights
        - Call to action
"""
)

linkedin_chain = linkedin_prompt | llm

linkedin_post = linkedin_chain.invoke({
    "listing": result.content
})

print(linkedin_post.content)

🌟 **Exciting Investment Opportunity: Prime Retail Center in Thriving Austin Market!** 🌟

Are you looking to diversify your portfolio with a high-performing asset in one of America's fastest-growing cities? Look no further! We're thrilled to present this outstanding investment opportunity nestled in the heart of Austin, Texas.

🌐 **Property Details:**  
**Location:** 123 Main Street, Austin, TX  
**Listing Price:** $8,500,000  
**Size:** 25,000 sq. ft. Retail Center  
**Current Occupancy:** 95%  

### **Key Highlights:**
- **Consistent Cash Flow:** With a 95% occupancy rate and a strong mix of reputable tenants, this retail center promises steady income in a bustling market.
- **Unbeatable Anchor Tenant:** Featuring a high-performing Starbucks, the property is positioned for success, attracting consistent foot traffic and enhancing tenant visibility.
- **Thriving Demographics:** Located in a rapidly developing area with an average household income of $145,000, this retail center caters 

In [12]:
# Generate a commercial real estate outreach email for the above listing using the generated description
email_prompt = PromptTemplate(
    input_variables=["listing"],
    template="""
        Write a commercial real estate outreach email.

        Property Details:
        {listing}

        Requirements:
        - Subject line
        - Short email
        - Strong call to action
"""
)

email_chain = email_prompt | llm

email = email_chain.invoke({
    "listing": result.content
})

print(email.content)

**Subject:** Invest in a Prime Retail Center in Thriving Austin!

Hi [Recipient's Name],

I hope this email finds you well! I wanted to share an incredible investment opportunity that has just hit the market: a **25,000 square foot retail center** at **123 Main Street, Austin, TX**, listed at **$8,500,000**. 

With a remarkable **95% occupancy rate** and a strong anchor tenant (Starbucks), this property offers consistent cash flow in a neighborhood with an average household income of **$145,000**. Its prime location, with **42,000 vehicles passing daily**, ensures high visibility and foot traffic.

Austin is one of the fastest-growing cities in the U.S., and this retail center is perfectly positioned to capitalize on its booming economy.

**Don’t miss out on this opportunity!** For more details or to schedule a private viewing, please reach out to me directly at [Your Phone Number] or [Your Email Address].

Looking forward to hearing from you!

Best regards,

[Your Name]  
[Your Broker

In [13]:
# Parse Output into JSON
from pydantic import BaseModel
from langchain_core.output_parsers import PydanticOutputParser

class PropertySummary(BaseModel):
    property_type: str
    city: str
    price: float
    key_tenants: list[str]

parser = PydanticOutputParser(
    pydantic_object=PropertySummary
)

prompt = PromptTemplate(
    template="""
        Extract information from the property.

        {format_instructions}

        Property:
        {property}
    """,
    input_variables=["property"],
    partial_variables={
        "format_instructions":
        parser.get_format_instructions()
    }
)

chain = prompt | llm | parser

result = chain.invoke({
    "property": property_info
})

print(result)

property_type='Retail Center' city='Austin' price=8500000.0 key_tenants=['Starbucks']
